**GUIDE**
-----------------------------------------------------------------------------
**Custom_Market_Text**: [Macro Event / Currency] + [Strong Directional Verb] + [Financial Context/Impact]

------------------------------------------------------------------------------
**Timeframe:**
*   15m
*   1h
*   4h
*   1d
------------------------------------------------------------------------------
**Target_Pairs:**
1. EURUSD=X
2. USDJPY=X
3. GBPUSD=X
4. AUDUSD=X
5. USDCAD=X
6. USDCHF=X
7. NZDUSD=X
8. EURJPY=X
9. GBPJPY=X
10. EURGBP=X
11. EURCHF=X
12. AUDJPY=X
13. GBPAUD=X
14. EURAUD=X
15. JPYCHF=X
16. USDMXN=X
17. USDZAR=X
18. USDSGD=X
19. USDHKD=X
20. USDPLN=X







In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import datetime
import requests
import warnings
import logging
import transformers
import os
import joblib
import concurrent.futures

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
transformers.logging.set_verbosity_error() # Suppresses FinBERT load warnings

#@title 📈 Yuji's ForEx Market Analyzer and Trends Predictor - Machine Learning Model Pipeline { display-mode: "form" }
News_API_Key = "" #@param {type:"string"}
Target_Pairs = "EURUSD=X, USDJPY=X, GBPUSD=X, AUDUSD=X, USDCAD=X, USDCHF=X, NZDUSD=X, EURJPY=X, GBPJPY=X, EURGBP=X, EURCHF=X, AUDJPY=X, GBPAUD=X, EURAUD=X, JPYCHF=X, USDMXN=X, USDZAR=X, USDSGD=X, USDHKD=X, USDPLN=X" #@param {type:"string"}
Timeframe = "4h" #@param ["1d", "1h", "4h", "15m"]
Prediction_Horizon_Bars = 20 #@param {type:"slider", min:1, max:24, step:1}
Custom_Market_Text = "" #@param {type:"string"}

# ==========================================
# Section 0: For Configurations & Settings
# ==========================================
class Config:
    PAIRS = [p.strip() for p in Target_Pairs.split(',')]
    HORIZON_BARS = Prediction_Horizon_Bars
    INTERVAL = Timeframe
    NEWS_API_KEY = News_API_Key

    # 0.1. PRO UPGRADE: Dynamic Sequence Lengths
    if Timeframe == '1d': SEQ_LENGTH = 10
    elif Timeframe == '4h': SEQ_LENGTH = 30 # Remembers 5 days of 4H data
    elif Timeframe == '1h': SEQ_LENGTH = 48 # Remembers 48 hours of 1H data
    else: SEQ_LENGTH = 24

    # 0.2. PRO UPGRADE: Checkpointing Directory
    MODEL_DIR = "./forex_models"

    # 0.3. LSTM Hyperparameters
    LSTM_HIDDEN = 64
    LSTM_LAYERS = 2
    EPOCHS = 15 # Reduced slightly for faster fine-tuning
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001

    # 0.4. Risk Management Settings
    SL_ATR_MULTIPLIER = 1.5
    TP_ATR_MULTIPLIER = 3.0
    MIN_WIN_RATE = 55.0 # Hard Gate threshold

os.makedirs(Config.MODEL_DIR, exist_ok=True)

# ==========================================
# Section 1: Data Ingestion Module
# ==========================================
class DataFetcher:
    def __init__(self, api_key=None):
        self.api_key = api_key

    def fetch_ohlcv(self, ticker, start_date, end_date, interval=Config.INTERVAL):
        try:
            data = yf.download(ticker, start=start_date, end=end_date, interval=interval, progress=False)
            if data.empty: return pd.DataFrame()

            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)

            data.index = pd.to_datetime(data.index).tz_localize(None)
            df = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
            for col in df.columns: df[col] = pd.to_numeric(df[col].squeeze(), errors='coerce')
            return df
        except Exception:
            return pd.DataFrame()

    def fetch_macro_data(self, start_date, end_date):
        try:
            macro_data = yf.download(['^VIX', '^TNX'], start=start_date, end=end_date, interval=Config.INTERVAL, progress=False)
            df = pd.DataFrame()
            if isinstance(macro_data.columns, pd.MultiIndex):
                df['VIX'] = macro_data['Close']['^VIX']
                df['US_10Y_Yield'] = macro_data['Close']['^TNX']
            else:
                df['VIX'] = macro_data.get('^VIX', 0)
                df['US_10Y_Yield'] = macro_data.get('^TNX', 0)

            df.index = pd.to_datetime(df.index).tz_localize(None)
            df.ffill(inplace=True)
            return df
        except Exception:
            return pd.DataFrame()

    def fetch_news_sentiment(self, query, start_date, end_date):
        def inject_custom_text(df):
            if Custom_Market_Text.strip():
                custom_row = pd.DataFrame([{'Date': pd.to_datetime(end_date).date(), 'Headline': Custom_Market_Text}])
                if df.empty: return custom_row
                return pd.concat([df, custom_row], ignore_index=True)
            return df

        if self.api_key == "YOUR_NEWS_API_KEY" or not self.api_key:
            dates = pd.date_range(start=start_date, end=end_date, freq='D')
            mock_news = pd.DataFrame({'Date': dates, 'Headline': [f"Mock market update {query}" for _ in dates]})
            return inject_custom_text(mock_news)

        url = f"https://newsapi.org/v2/everything?q={query}&from={start_date}&to={end_date}&sortBy=relevancy&apiKey={self.api_key}"
        try:
            response = requests.get(url)
            data = response.json()
            if data.get('status') != 'ok': raise Exception(data.get('message'))
            headlines = [{'Date': pd.to_datetime(a['publishedAt']).date(), 'Headline': a['title']} for a in data.get('articles', [])]
            df = pd.DataFrame(headlines)
            if not df.empty: df['Date'] = pd.to_datetime(df['Date']).dt.date
            return inject_custom_text(df)
        except Exception:
            return inject_custom_text(pd.DataFrame())

# ==========================================
# Section 2: Sentiment Analysis Module
# ==========================================
class SentimentAnalyzer:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
        self.model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        self.nlp = pipeline("sentiment-analysis", model=self.model, tokenizer=self.tokenizer, device=0 if torch.cuda.is_available() else -1)

    def analyze_dataframe(self, df):
        if df.empty: return pd.DataFrame(columns=['Date', 'Sentiment_Score']).set_index('Date')
        try:
            sentiments = self.nlp(df['Headline'].tolist(), truncation=True, max_length=512)
            scores = [res['score'] if res['label'] == 'positive' else -res['score'] if res['label'] == 'negative' else 0.0 for res in sentiments]
            df['Sentiment_Score'] = scores
            daily_sentiment = df.groupby('Date')['Sentiment_Score'].mean().reset_index()
            daily_sentiment.set_index('Date', inplace=True)
            return daily_sentiment
        except Exception:
            return pd.DataFrame(columns=['Date', 'Sentiment_Score']).set_index('Date')

# ==========================================
# Section 3: Feature Engineering Module
# ==========================================
class FeatureEngineer:
    @staticmethod
    def calculate_indicators(df):
        df = df.copy()
        close = df['Close'].squeeze()

        # 3.1. Volume Weighted Average Price (VWAP) - Resets daily
        df['Date'] = df.index.date
        df['Typical_Price'] = (df['High'] + df['Low'] + df['Close']) / 3
        df['VWAP'] = df.groupby('Date').apply(lambda x: (x['Typical_Price'] * x['Volume']).cumsum() / (x['Volume'].cumsum() + 1e-9)).reset_index(level=0, drop=True)
        df.drop(columns=['Date', 'Typical_Price'], inplace=True)

        # 3.2. RSI & MACD
        delta = close.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['RSI'] = 100 - (100 / (1 + rs))

        exp1 = close.ewm(span=12, adjust=False).mean()
        exp2 = close.ewm(span=26, adjust=False).mean()
        df['MACD'] = exp1 - exp2

        # 3.3. Bollinger Bands & ATR
        df['BB_Mid'] = close.rolling(window=20).mean()
        std_dev = close.rolling(window=20).std()
        df['BB_Upper'] = df['BB_Mid'] + (std_dev * 2)
        df['BB_Lower'] = df['BB_Mid'] - (std_dev * 2)

        true_range = pd.concat([df['High'] - df['Low'], np.abs(df['High'] - close.shift()), np.abs(df['Low'] - close.shift())], axis=1).max(axis=1)
        df['ATR'] = true_range.rolling(window=14).mean()

        # 3.4. Trend Indicators
        df['EMA_20'] = close.ewm(span=20, adjust=False).mean()
        df['EMA_50'] = close.ewm(span=50, adjust=False).mean()

        # 3.5. Target Variable
        df['Target_Return'] = close.shift(-Config.HORIZON_BARS) / close - 1
        return df

    @staticmethod
    def merge_features(price_df, sentiment_df, macro_df):
        df = price_df.copy()
        if not macro_df.empty:
            df = df.join(macro_df, how='left')
            df['VIX'] = df['VIX'].ffill().fillna(0)
            df['US_10Y_Yield'] = df['US_10Y_Yield'].ffill().fillna(0)

        # 3.6. Normalize dates to datetime64[ns] (to avoid Pandas merge type mismatch)
        df['Date_Only'] = pd.to_datetime(df.index).normalize()
        sentiment_df.index = pd.to_datetime(sentiment_df.index).normalize()

        df = df.merge(sentiment_df, left_on='Date_Only', right_index=True, how='left')
        df.drop(columns=['Date_Only'], inplace=True)
        df['Sentiment_Score'] = df['Sentiment_Score'].ffill().fillna(0)
        df.dropna(inplace=True)
        return df

# ==========================================
# Section 4: Model Architecture
# ==========================================
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_layer_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_layer_size, num_layers, batch_first=True, dropout=0.2)
        self.linear = nn.Linear(hidden_layer_size, 1)

    def forward(self, input_seq):
        lstm_out, _ = self.lstm(input_seq)
        return self.linear(lstm_out[:, -1, :])

class EnsemblePredictor:
    def __init__(self, input_size, ticker):
        self.ticker = ticker
        self.xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
        self.lstm_model = LSTMModel(input_size=input_size, hidden_layer_size=Config.LSTM_HIDDEN, num_layers=Config.LSTM_LAYERS)
        self.scaler_X = StandardScaler()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.lstm_model.to(self.device)
        self.target_std = 0.01

        # 4.1. Checkpoint paths
        self.xgb_path = os.path.join(Config.MODEL_DIR, f"{ticker}_xgb.json")
        self.lstm_path = os.path.join(Config.MODEL_DIR, f"{ticker}_lstm.pt")
        self.scaler_path = os.path.join(Config.MODEL_DIR, f"{ticker}_scaler.pkl")

    def _prepare_lstm_data(self, X, y=None):
        Xs, ys = [], []
        for i in range(len(X) - Config.SEQ_LENGTH):
            Xs.append(X[i:(i + Config.SEQ_LENGTH)])
            if y is not None: ys.append(y[i + Config.SEQ_LENGTH])
        if y is not None:
            return torch.tensor(np.array(Xs), dtype=torch.float32).to(self.device), torch.tensor(np.array(ys), dtype=torch.float32).unsqueeze(1).to(self.device)
        return torch.tensor(np.array(Xs), dtype=torch.float32).to(self.device)

    def train(self, X, y, tune_hyperparameters=False):
        # 4.2. Model Checkpointing (Save & Load logic)
        model_exists = os.path.exists(self.xgb_path) and os.path.exists(self.lstm_path) and os.path.exists(self.scaler_path)

        if model_exists:
            self.scaler_X = joblib.load(self.scaler_path)
            self.xgb_model.load_model(self.xgb_path)
            self.lstm_model.load_state_dict(torch.load(self.lstm_path, map_location=self.device))
            X_scaled = self.scaler_X.transform(X) # 4.3. Transform using existing scaler
        else:
            X_scaled = self.scaler_X.fit_transform(X)
            joblib.dump(self.scaler_X, self.scaler_path)

        y_np = y.values
        X_xgb, y_aligned = X_scaled[Config.SEQ_LENGTH:], y_np[Config.SEQ_LENGTH:]
        self.target_std = np.std(y_aligned) if np.std(y_aligned) > 0 else 1e-5

        sample_weights = np.ones(len(y_aligned))
        recent_days = min(28, len(sample_weights))
        sample_weights[-recent_days:] = 2.0

        # 4.4. Train or Fine-tune XGBoost
        if tune_hyperparameters and not model_exists:
            param_grid = {'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1], 'n_estimators': [50, 100, 150]}
            search = RandomizedSearchCV(self.xgb_model, param_grid, n_iter=5, cv=3, random_state=42)
            search.fit(X_xgb, y_aligned, sample_weight=sample_weights)
            self.xgb_model = search.best_estimator_
        else:
            self.xgb_model.fit(X_xgb, y_aligned, sample_weight=sample_weights) # 4.5. Fine-tune

        self.xgb_model.save_model(self.xgb_path)

        # 4.6. Train or Fine-tune LSTM
        X_seq, y_seq = self._prepare_lstm_data(X_scaled, y_np)
        dataset = TensorDataset(X_seq, y_seq)
        loader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=True)

        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.lstm_model.parameters(), lr=Config.LEARNING_RATE)

        self.lstm_model.train()
        epochs_to_run = Config.EPOCHS if not model_exists else 3 # 4.7. Light fine-tuning if pre-loaded
        for epoch in range(epochs_to_run):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                loss = criterion(self.lstm_model(batch_X), batch_y)
                loss.backward()
                optimizer.step()

        torch.save(self.lstm_model.state_dict(), self.lstm_path)

    def predict(self, X):
        X_scaled = self.scaler_X.transform(X)
        X_xgb = X_scaled[Config.SEQ_LENGTH:]
        xgb_preds = self.xgb_model.predict(X_xgb)

        X_seq = self._prepare_lstm_data(X_scaled)
        self.lstm_model.eval()
        with torch.no_grad(): lstm_preds = self.lstm_model(X_seq).cpu().numpy().flatten()

        ensemble_preds = (xgb_preds + lstm_preds) / 2
        confidence = np.exp(-np.abs(xgb_preds - lstm_preds) / self.target_std)
        return ensemble_preds, confidence

# ==========================================
# Section 5: Pipeline & Backtester
# ==========================================
class TradingPipeline:
    def __init__(self):
        self.fetcher = DataFetcher(api_key=Config.NEWS_API_KEY)
        self.sentiment_analyzer = SentimentAnalyzer()

    def prepare_data(self, ticker, start_date, end_date):
        price_df = self.fetcher.fetch_ohlcv(ticker, start_date, end_date)
        if price_df.empty: return None, None

        # 5.1. Multi-Timeframe Analysis (MTFA)
        if Config.INTERVAL != '1d':
            daily_df = self.fetcher.fetch_ohlcv(ticker, start_date, end_date, interval='1d')
            if not daily_df.empty:
                daily_df['Daily_EMA_50'] = daily_df['Close'].ewm(span=50, adjust=False).mean()

                # 5.1.a. Normalize dates to datetime64[ns] here as well
                daily_df.index = pd.to_datetime(daily_df.index).normalize()
                price_df['Date_Only'] = pd.to_datetime(price_df.index).normalize()

                price_df = price_df.merge(daily_df[['Daily_EMA_50']], left_on='Date_Only', right_index=True, how='left')
                price_df.drop(columns=['Date_Only'], inplace=True)
                price_df['Daily_EMA_50'] = price_df['Daily_EMA_50'].ffill()
                # 1 = Bullish Macro Trend, -1 = Bearish Macro Trend
                price_df['Macro_Trend_Alignment'] = np.where(price_df['Close'] > price_df['Daily_EMA_50'], 1, -1)
            else:
                price_df['Macro_Trend_Alignment'] = 0

        macro_df = self.fetcher.fetch_macro_data(start_date, end_date)

        query = ticker[:3]
        end_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d').date()
        news_start_dt = max(datetime.datetime.strptime(start_date, '%Y-%m-%d').date(), end_dt - datetime.timedelta(days=28))
        news_df = self.fetcher.fetch_news_sentiment(query, news_start_dt.strftime('%Y-%m-%d'), end_date)
        sentiment_df = self.sentiment_analyzer.analyze_dataframe(news_df)

        features_df = FeatureEngineer.calculate_indicators(price_df)
        final_df = FeatureEngineer.merge_features(features_df, sentiment_df, macro_df)

        return final_df, price_df

    def walk_forward_backtest(self, df, ticker):
        features, target = df.drop(columns=['Target_Return']), df['Target_Return']
        tscv = TimeSeriesSplit(n_splits=3)
        correct_directions, total_predictions = 0, 0

        for train_idx, test_idx in tscv.split(features):
            if len(train_idx) <= Config.SEQ_LENGTH or len(test_idx) <= Config.SEQ_LENGTH: continue
            X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
            y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

            # Temporary model for backtesting (don't load checkpoints)
            model = EnsemblePredictor(input_size=X_train.shape[1], ticker=f"temp_bt_{ticker}")
            model.train(X_train, y_train, tune_hyperparameters=False)

            preds, _ = model.predict(X_test)
            y_test_aligned = y_test.values[Config.SEQ_LENGTH:]
            correct_directions += np.sum(np.sign(y_test_aligned) == np.sign(preds))
            total_predictions += len(y_test_aligned)

        return round((correct_directions / total_predictions * 100), 2) if total_predictions > 0 else 0

    def process_single_pair(self, pair, start_date, end_date):
        """Processes a single currency pair. Designed for Parallel Processing."""
        logging.info(f"Processing {pair}...")
        df, raw_price_df = self.prepare_data(pair, start_date, end_date)
        if df is None or len(df) < Config.SEQ_LENGTH + 30: return None

        win_rate = self.walk_forward_backtest(df, pair)

        X, y = df.drop(columns=['Target_Return']), df['Target_Return']
        model = EnsemblePredictor(input_size=X.shape[1], ticker=pair)
        model.train(X, y, tune_hyperparameters=True)

        recent_X = X.iloc[-Config.SEQ_LENGTH - 1:]
        pred_return, confidence = model.predict(recent_X)

        current_price = raw_price_df['Close'].iloc[-1]
        current_atr = df['ATR'].iloc[-1]
        predicted_move = pred_return[-1]

        action = "BUY" if predicted_move > 0 else "SELL"

        # Hard Gating Logic (Safety Switch)
        if win_rate < Config.MIN_WIN_RATE:
            action = "NO TRADE"
            take_profit, stop_loss = 0.0, 0.0
        elif action == "BUY":
            take_profit = current_price + (current_atr * Config.TP_ATR_MULTIPLIER)
            stop_loss = current_price - (current_atr * Config.SL_ATR_MULTIPLIER)
        else:
            take_profit = current_price - (current_atr * Config.TP_ATR_MULTIPLIER)
            stop_loss = current_price + (current_atr * Config.SL_ATR_MULTIPLIER)

        return {
            'Currency_Pair': pair,
            'Action': action,
            'Win_Rate_%': win_rate,
            'Confidence': round(confidence[-1], 4),
            'Pred_Return_%': round(predicted_move * 100, 4),
            'Current_Price': round(current_price, 5),
            'Take_Profit': round(take_profit, 5),
            'Stop_Loss': round(stop_loss, 5)
        }

    def predict_top_gainers(self):
        end_date = datetime.date.today() + datetime.timedelta(days=1)
        start_date = end_date - datetime.timedelta(days=365 if Config.INTERVAL == '1d' else 59)

        predictions = []
        start_str, end_str = start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')

        # Parallel Processing (Drastically cuts runtime)
        with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
            futures = [executor.submit(self.process_single_pair, pair, start_str, end_str) for pair in Config.PAIRS]
            for future in concurrent.futures.as_completed(futures):
                result = future.result()
                if result: predictions.append(result)

        results_df = pd.DataFrame(predictions)
        if not results_df.empty:
            results_df.sort_values(by=['Action', 'Confidence'], ascending=[True, False], inplace=True) # Puts 'BUY/SELL' at the top, 'NO TRADE' at the bottom
            results_df.reset_index(drop=True, inplace=True)
        return results_df

# ==========================================
# Section 6: Execution Script
# ==========================================
if __name__ == "__main__":
    print("="*80)
    print(" YUJI'S FOREX MARKET ANALYZER AND TRENDS PREDICTOR (MTFA + MLOps) ")
    print("="*80)

    pipeline_manager = TradingPipeline()
    try:
        ranked_pairs = pipeline_manager.predict_top_gainers()
        print("\n" + "="*80)
        print(f" FINAL RANKING & TRADING SIGNALS ({Config.HORIZON_BARS} BARS AHEAD | {Config.INTERVAL.upper()}) ")
        print("="*80)
        print(ranked_pairs.to_string(index=False))
        print(f"\nNote: All trades below {Config.MIN_WIN_RATE}% Win Rate are hard-gated to 'NO TRADE'.")

    except Exception as e:
        logging.error(f"Pipeline failed during execution: {e}")

 YUJI'S FOREX MARKET ANALYZER AND TRENDS PREDICTOR (MTFA + MLOps) 


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]


 FINAL RANKING & TRADING SIGNALS (20 BARS AHEAD | 4H) 
Currency_Pair   Action  Win_Rate_%  Confidence  Pred_Return_%  Current_Price  Take_Profit  Stop_Loss
     USDJPY=X      BUY       65.30      0.7455         0.0409      159.33299    159.89615  159.05141
     USDZAR=X      BUY       68.06      0.6525         1.0137       16.52230     16.80411   16.38140
     EURCHF=X      BUY       69.44      0.2225         0.0539        0.91935      0.92437    0.91684
     GBPAUD=X NO TRADE       51.39      0.8829         0.3132        1.89247      0.00000    0.00000
     EURAUD=X NO TRADE       48.93      0.8379         0.0095        1.63883      0.00000    0.00000
     GBPJPY=X NO TRADE       46.11      0.8287        -0.0778        0.71508      0.00000    0.00000
     NZDUSD=X NO TRADE       45.82      0.8212         0.0229        0.71508      0.00000    0.00000
     USDMXN=X NO TRADE       27.78      0.7974         0.5079       17.39350      0.00000    0.00000
     USDCHF=X NO TRADE       47.65 